In [1]:
import pandas as pd
import numpy as np
import random

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

# Generate 10,000 customer records
n_customers = 10000

# Generate customer IDs
customer_ids = [f"CUST_{i:05d}" for i in range(1, n_customers + 1)]

# Generate basic demographics
genders = np.random.choice(['Male', 'Female'], n_customers, p=[0.51, 0.49])
ages = np.random.normal(45, 15, n_customers).astype(int)
ages = np.clip(ages, 18, 80)

# Generate tenure (months with company)
# New customers more likely to churn
tenure_weights = [0.3, 0.25, 0.2, 0.15, 0.1]
tenure_ranges = [(0, 12), (12, 24), (24, 36), (36, 48), (48, 72)]
tenure = []
for _ in range(n_customers):
    range_idx = np.random.choice(len(tenure_ranges), p=tenure_weights)
    min_val, max_val = tenure_ranges[range_idx]
    tenure.append(np.random.randint(min_val, max_val + 1))

# Generate contract types (correlated with churn)
contract_types = []
for t in tenure:
    if t < 12:
        contract_types.append(np.random.choice(['Month-to-month', 'One year', 'Two year'], p=[0.7, 0.25, 0.05]))
    elif t < 24:
        contract_types.append(np.random.choice(['Month-to-month', 'One year', 'Two year'], p=[0.5, 0.35, 0.15]))
    else:
        contract_types.append(np.random.choice(['Month-to-month', 'One year', 'Two year'], p=[0.3, 0.4, 0.3]))

# Generate payment methods
payment_methods = np.random.choice(['Electronic check', 'Mailed check', 'Bank transfer (automatic)', 'Credit card (automatic)'], 
                                 n_customers, p=[0.35, 0.15, 0.25, 0.25])

# Generate internet service
internet_services = np.random.choice(['DSL', 'Fiber optic', 'No'], n_customers, p=[0.4, 0.45, 0.15])

# Generate monthly charges based on services
monthly_charges = []
total_charges = []

for i in range(n_customers):
    base_charge = 20
    
    # Add charges based on internet service
    if internet_services[i] == 'DSL':
        base_charge += np.random.normal(25, 5)
    elif internet_services[i] == 'Fiber optic':
        base_charge += np.random.normal(45, 8)
    
    # Add random variation
    monthly_charge = base_charge + np.random.normal(0, 10)
    monthly_charge = max(20, min(120, monthly_charge))
    monthly_charges.append(round(monthly_charge, 2))
    
    # Calculate total charges
    total_charge = monthly_charge * tenure[i] + np.random.normal(0, 50)
    total_charge = max(20, total_charge)
    total_charges.append(round(total_charge, 2))

# Generate binary features
senior_citizens = np.random.choice([0, 1], n_customers, p=[0.84, 0.16])
partners = np.random.choice(['Yes', 'No'], n_customers, p=[0.52, 0.48])
dependents = np.random.choice(['Yes', 'No'], n_customers, p=[0.32, 0.68])
phone_services = np.random.choice(['Yes', 'No'], n_customers, p=[0.91, 0.09])
paperless_billing = np.random.choice(['Yes', 'No'], n_customers, p=[0.63, 0.37])

# Generate service features (correlated with internet service)
def generate_service_feature(internet_service):
    if internet_service == 'No':
        return 'No internet service'
    else:
        return np.random.choice(['Yes', 'No'], p=[0.4, 0.6])

online_security = [generate_service_feature(svc) for svc in internet_services]
online_backup = [generate_service_feature(svc) for svc in internet_services]
device_protection = [generate_service_feature(svc) for svc in internet_services]
tech_support = [generate_service_feature(svc) for svc in internet_services]
streaming_tv = [generate_service_feature(svc) for svc in internet_services]
streaming_movies = [generate_service_feature(svc) for svc in internet_services]

# Generate multiple lines (correlated with phone service)
multiple_lines = []
for phone in phone_services:
    if phone == 'No':
        multiple_lines.append('No phone service')
    else:
        multiple_lines.append(np.random.choice(['Yes', 'No'], p=[0.45, 0.55]))

# Generate churn based on realistic business logic
churn_probability = []
for i in range(n_customers):
    prob = 0.1  # Base probability
    
    # Tenure effect (strongest predictor)
    if tenure[i] < 6:
        prob += 0.4
    elif tenure[i] < 12:
        prob += 0.3
    elif tenure[i] < 24:
        prob += 0.2
    elif tenure[i] < 36:
        prob += 0.1
    
    # Contract type effect
    if contract_types[i] == 'Month-to-month':
        prob += 0.25
    elif contract_types[i] == 'One year':
        prob += 0.1
    
    # Payment method effect
    if payment_methods[i] == 'Electronic check':
        prob += 0.15
    
    # Monthly charges effect
    if monthly_charges[i] > 80:
        prob += 0.1
    
    # Senior citizen effect
    if senior_citizens[i] == 1:
        prob += 0.05
    
    # Internet service effect
    if internet_services[i] == 'Fiber optic':
        prob += 0.05
    
    # Support services effect
    if online_security[i] == 'No':
        prob += 0.05
    if tech_support[i] == 'No':
        prob += 0.05
    
    # Family effect (reduces churn)
    if partners[i] == 'Yes':
        prob -= 0.05
    if dependents[i] == 'Yes':
        prob -= 0.08
    
    # Cap probability
    prob = min(0.8, max(0.05, prob))
    churn_probability.append(prob)

# Generate actual churn based on probabilities
churn = [np.random.choice(['Yes', 'No'], p=[prob, 1-prob]) for prob in churn_probability]

# Add data quality issues - some total_charges as strings with spaces
total_charges_str = []
for i, charge in enumerate(total_charges):
    if i < 50:  # First 50 customers have this data quality issue
        total_charges_str.append(' ')
    else:
        total_charges_str.append(str(charge))

# Create DataFrame
df = pd.DataFrame({
    'customer_id': customer_ids,
    'gender': genders,
    'age': ages,
    'senior_citizen': senior_citizens,
    'partner': partners,
    'dependents': dependents,
    'tenure': tenure,
    'phone_service': phone_services,
    'multiple_lines': multiple_lines,
    'internet_service': internet_services,
    'online_security': online_security,
    'online_backup': online_backup,
    'device_protection': device_protection,
    'tech_support': tech_support,
    'streaming_tv': streaming_tv,
    'streaming_movies': streaming_movies,
    'contract_type': contract_types,
    'paperless_billing': paperless_billing,
    'payment_method': payment_methods,
    'monthly_charges': monthly_charges,
    'total_charges': total_charges_str,
    'churn': churn
})

# Save to CSV
df.to_csv('telecom_customer_data.csv', index=False)

# Display basic statistics
print("Dataset created successfully!")
print(f"Shape: {df.shape}")
print(f"Churn rate: {df['churn'].value_counts(normalize=True)['Yes']:.2%}")
print("\nFirst 5 rows:")
print(df.head())
print("\nDataset info:")
print(df.info())
print("\nChurn distribution:")
print(df['churn'].value_counts())
print("\nSample statistics:")
print(df[['age', 'tenure', 'monthly_charges']].describe())

# Show some data quality issues for practice
print("\nData quality issues to handle:")
print(f"Missing total_charges (spaces): {sum(1 for x in total_charges_str if x.strip() == '')}")
print(f"Unique contract types: {df['contract_type'].unique()}")
print(f"Unique payment methods: {df['payment_method'].unique()}")

Dataset created successfully!
Shape: (10000, 22)
Churn rate: 49.51%

First 5 rows:
  customer_id  gender  age  senior_citizen partner dependents  tenure  \
0  CUST_00001    Male   22               0      No         No       3   
1  CUST_00002  Female   28               0     Yes         No      21   
2  CUST_00003  Female   50               0      No         No       7   
3  CUST_00004  Female   27               0      No        Yes      17   
4  CUST_00005    Male   61               0      No         No      24   

  phone_service    multiple_lines internet_service  ...    device_protection  \
0           Yes                No               No  ...  No internet service   
1            No  No phone service      Fiber optic  ...                   No   
2           Yes               Yes      Fiber optic  ...                   No   
3           Yes               Yes      Fiber optic  ...                   No   
4           Yes               Yes              DSL  ...                  Yes  

In [2]:
# Data overview
print(df.info())
print(df.describe())
print(df.isnull().sum())

# Target variable distribution
churn_rate = df['churn'].value_counts(normalize=True)
print(f"Churn rate: {churn_rate['Yes']:.2%}")

# Numerical features analysis
numerical_features = ['age', 'tenure', 'monthly_charges', 'total_charges']
for feature in numerical_features:
    # Distribution by churn status
    churned = df[df['churn'] == 'Yes'][feature]
    retained = df[df['churn'] == 'No'][feature]
    
    # Statistical tests
    from scipy.stats import ttest_ind
    stat, p_value = ttest_ind(churned, retained)

# Categorical features analysis
categorical_features = ['contract_type', 'payment_method', 'internet_service']
for feature in categorical_features:
    # Cross-tabulation with churn
    ct = pd.crosstab(df[feature], df['churn'], normalize='index')
    print(ct)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 22 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   customer_id        10000 non-null  object 
 1   gender             10000 non-null  object 
 2   age                10000 non-null  int64  
 3   senior_citizen     10000 non-null  int64  
 4   partner            10000 non-null  object 
 5   dependents         10000 non-null  object 
 6   tenure             10000 non-null  int64  
 7   phone_service      10000 non-null  object 
 8   multiple_lines     10000 non-null  object 
 9   internet_service   10000 non-null  object 
 10  online_security    10000 non-null  object 
 11  online_backup      10000 non-null  object 
 12  device_protection  10000 non-null  object 
 13  tech_support       10000 non-null  object 
 14  streaming_tv       10000 non-null  object 
 15  streaming_movies   10000 non-null  object 
 16  contract_type      1000

TypeError: unsupported operand type(s) for /: 'str' and 'int'

In [3]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Preprocessing
X = df.drop(['customer_id', 'churn'], axis=1)
y = df['churn']

# Encode categorical variables
X_encoded = pd.get_dummies(X, drop_first=True)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

# Train decision tree
dt = DecisionTreeClassifier(
    max_depth=5,           # Prevent overfitting
    min_samples_split=100, # Minimum samples to split
    min_samples_leaf=50,   # Minimum samples in leaf
    class_weight='balanced' # Handle class imbalance
)

dt.fit(X_train, y_train)

# Feature importance
feature_importance = pd.DataFrame({
    'feature': X_encoded.columns,
    'importance': dt.feature_importances_
}).sort_values('importance', ascending=False)

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Feature scaling for logistic regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Logistic regression with regularization
lr = LogisticRegression(
    class_weight='balanced',
    C=1.0,  # Regularization strength
    penalty='l2',
    random_state=42
)

lr.fit(X_train_scaled, y_train)

# Coefficient interpretation
coefficients = pd.DataFrame({
    'feature': X_encoded.columns,
    'coefficient': lr.coef_[0],
    'odds_ratio': np.exp(lr.coef_[0])
}).sort_values('coefficient', key=abs, ascending=False)

In [6]:
# Cost-benefit analysis
def calculate_profit(y_true, y_pred, threshold=0.5):
    # Convert probabilities to predictions
    predictions = (y_pred > threshold).astype(int)
    
    # Calculate confusion matrix components
    tp = sum((y_true == 1) & (predictions == 1))  # Correctly identified churners
    fp = sum((y_true == 0) & (predictions == 1))  # False alarms
    fn = sum((y_true == 1) & (predictions == 0))  # Missed churners
    
    # Business metrics
    retention_cost = (tp + fp) * 50  # Cost of retention campaigns
    retention_revenue = tp * 0.4 * 2400  # Successful retentions * CLV
    churn_cost = fn * 2400  # Lost customers
    
    profit = retention_revenue - retention_cost - churn_cost
    return profit

# Optimize threshold for maximum profit
thresholds = np.arange(0.1, 0.9, 0.05)
profits = [calculate_profit(y_test, y_pred_proba[:, 1], t) for t in thresholds]
optimal_threshold = thresholds[np.argmax(profits)]

NameError: name 'y_pred_proba' is not defined

In [9]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier


from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

class ChurnPreprocessor:
    def __init__(self):
        self.scaler = StandardScaler()
        self.label_encoders = {}
        self.imputer = SimpleImputer(strategy='median')
        
    def fit_transform(self, df):
        # Handle missing values
        df = df.copy()
        
        # Convert total_charges to numeric (handle spaces)
        df['total_charges'] = pd.to_numeric(df['total_charges'], errors='coerce')
        
        # Impute missing values
        numerical_cols = ['age', 'tenure', 'monthly_charges', 'total_charges']
        df[numerical_cols] = self.imputer.fit_transform(df[numerical_cols])
        
        # Feature engineering
        df['charges_per_month'] = df['total_charges'] / (df['tenure'] + 1)
        df['is_new_customer'] = (df['tenure'] < 12).astype(int)
        df['high_value_customer'] = (df['monthly_charges'] > df['monthly_charges'].quantile(0.8)).astype(int)
        
        # Encode categorical variables
        categorical_cols = ['gender', 'contract_type', 'payment_method', 'internet_service']
        for col in categorical_cols:
            self.label_encoders[col] = LabelEncoder()
            df[col] = self.label_encoders[col].fit_transform(df[col])
        
        # Scale numerical features
        numerical_cols_extended = numerical_cols + ['charges_per_month']
        df[numerical_cols_extended] = self.scaler.fit_transform(df[numerical_cols_extended])
        
        return df

In [11]:
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.metrics import make_scorer, f1_score

# Define models to compare
models = {
    'Logistic Regression': LogisticRegression(class_weight='balanced'),
    'Random Forest': RandomForestClassifier(class_weight='balanced'),
    'XGBoost': XGBClassifier(scale_pos_weight=3)
}

# Cross-validation comparison
cv_results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='f1')
    cv_results[name] = {
        'mean': scores.mean(),
        'std': scores.std()
    }

# Hyperparameter tuning for best model
best_model = 'XGBoost'  # Based on CV results
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'scale_pos_weight': [1, 3, 5]
}

grid_search = GridSearchCV(
    XGBClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
best_model = grid_search.best_estimator_

/Users/yenokhakobyan/miniconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/yenokhakobyan/miniconda3/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:978: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/Users/yenokhakobyan/miniconda3/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 140, in __call__
    score = scorer._score(
            ^^^^^^^^^^^^^^
  File "/Users/yenokhakobyan/minico

AttributeError: 'super' object has no attribute '__sklearn_tags__'